# Complete-official-train fine-tuning

This notebook consumes one selected LR, reloads the original pretrained model,
trains on every official training image for 25 epochs, and then invokes the
common repository evaluator once on complete official validation.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration


In [ ]:
MODEL_ID = "rtdetrv2_l"
START_EXPENSIVE_STAGE = False
ALLOW_OVER_BUDGET_RUN = False
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

# Locked benchmark controls; do not edit for comparable runs.
FINAL_EPOCHS = 25
FINAL_SEED = 42
if SMOKE_TEST:
    START_EXPENSIVE_STAGE = False
assert FINAL_EPOCHS == 25 and FINAL_SEED == 42
assert PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS == 8


## Install and validate the selected model environment


In [ ]:
if IS_COLAB and not SMOKE_TEST:
    requirement = (
        "requirements-rtdetr-colab.txt"
        if MODEL_ID == "rtdetrv2_l"
        else "requirements-openmmlab-py310-cu118.txt"
    )
    if MODEL_ID != "rtdetrv2_l" and sys.version_info[:2] != (3, 10):
        raise RuntimeError(
            "OpenMMLab models require the documented Python 3.10 custom/local "
            "Colab runtime; the current hosted runtime is not supported."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", requirement],
        check=True,
    )
if MODEL_ID != "rtdetrv2_l":
    upstream = Path("/content/VMamba" if MODEL_ID == "faster_rcnn_vmamba_t" else "/content/mmdetection")
    if IS_COLAB and not upstream.joinpath(".git").is_dir() and not SMOKE_TEST:
        url = (
            "https://github.com/MzeroMiko/VMamba.git"
            if MODEL_ID == "faster_rcnn_vmamba_t"
            else "https://github.com/open-mmlab/mmdetection.git"
        )
        clone_command = ["git", "clone"]
        if MODEL_ID != "faster_rcnn_vmamba_t":
            clone_command.extend(["--depth", "1", "--branch", "v3.3.0"])
        clone_command.extend([url, str(upstream)])
        subprocess.run(clone_command, check=True)
    os.environ[
        "VMAMBA_ROOT" if MODEL_ID == "faster_rcnn_vmamba_t" else "MMDET_ROOT"
    ] = str(upstream)
    if MODEL_ID == "faster_rcnn_vmamba_t":
        subprocess.run(
            ["git", "-C", str(upstream), "checkout",
             "2ed52ead062a51a64521ed3871d52914bf532876"],
            check=True,
        )
        pretrained = paths.pretrained / "vmamba_t.pth"
        if not pretrained.is_file() and not SMOKE_TEST:
            raise FileNotFoundError(
                f"Place the verified official VMamba-T checkpoint at {pretrained}"
            )
        os.environ["VMAMBA_T_PRETRAINED"] = str(pretrained)
        if not SMOKE_TEST:
            try:
                import selective_scan_cuda
            except ImportError:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install",
                     str(upstream / "kernels" / "selective_scan"),
                     "--no-build-isolation"],
                    check=True,
                )
            import selective_scan_cuda
print("Environment selected for:", MODEL_ID)


## Load and validate the selected configuration


In [ ]:
import json
from src.training.lr_workflow import LRControlledBenchmark
from src.utils.serialization import read_yaml
from src.benchmark_status import (
    discover_model_status,
    find_resumable_final_run,
    find_selected_config,
    format_preflight_summary,
)

workflow = LRControlledBenchmark(REPO_DIR, DRIVE_ROOT)
workflow.prepare_manifests()
selected_path = find_selected_config(DRIVE_ROOT, MODEL_ID, REPO_DIR)
if selected_path is None:
    if SMOKE_TEST:
        print("No selected YAML in smoke mode; run notebook 12 on a GPU first.")
        selected = None
    else:
        raise FileNotFoundError(
            f"Selected LR YAML for {MODEL_ID} is missing. Complete notebook 12 first."
        )
else:
    selected = read_yaml(selected_path)
    assert selected["experiment"]["model_id"] == MODEL_ID
    assert selected["final_training"]["dataset"] == "complete_official_train"
    assert selected["final_training"]["restart_from_pretrained"] is True
    print(selected)


## Prove complete-train identity and validation exclusion


In [ ]:
from src.training.lr_search import assert_final_training_uses_official_train
assert_final_training_uses_official_train(workflow.manifest_dir)
summary = json.loads((workflow.manifest_dir / "split_summary.json").read_text())
print("Complete official train proof:", summary["sources"]["official_train"])
print("Split checks:", summary["verification"])
print("FULL OFFICIAL TRAINING SPLIT VERIFIED: YES")
if selected is not None:
    final = selected["final_training"]
    resumable = find_resumable_final_run(
        DRIVE_ROOT,
        MODEL_ID,
        selected_learning_rate=float(final["learning_rate"]),
    )
    calibration_path = paths.lr_search_checkpoints / MODEL_ID / "calibration.json"
    estimate = (
        workflow.workload_estimate(json.loads(calibration_path.read_text()))
        if calibration_path.exists()
        else None
    )
    try:
        import torch
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT DETECTED"
    except ImportError:
        gpu_name = "NOT DETECTED"
    print(format_preflight_summary({
        "model": MODEL_ID,
        "dataset_track": "2class",
        "mode": "FINAL FINE-TUNING",
        "train_manifest": workflow.manifest_dir / "official_full_train.json",
        "validation_manifest": workflow.manifest_dir / "official_validation.json",
        "training_images": summary["statistics"]["official_full_train.json"]["images"],
        "validation_images": summary["statistics"]["official_validation.json"]["images"],
        "full_official_train": "YES",
        "image_size": 640,
        "batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": 8,
        "learning_rate": final["learning_rate"],
        "epoch_budget": 25,
        "gpu": gpu_name,
        "estimated_runtime": f"{estimate['final_seconds'] / 3600:.2f} h" if estimate else "calculated after calibration",
        "output_directory": (
            resumable["run_dir"] if resumable
            else paths.final_checkpoints / MODEL_ID / "<new-run-id>"
        ),
        "resume_status": (
            f"AUTO-RESUME {resumable['run_id']}" if resumable else "NEW RUN"
        ),
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    }))


## Restart from pretrained and run final benchmark


In [ ]:
if START_EXPENSIVE_STAGE:
    if selected is None:
        raise RuntimeError("A selected LR configuration is required.")
    from src.notebook_utils import require_gpu, require_model_environment
    require_model_environment(
        "rtdetr" if MODEL_ID == "rtdetrv2_l" else "openmmlab"
    )
    require_gpu(MODEL_ID)
    final_manifest = workflow.run_final_training(
        MODEL_ID,
        selected_path,
        batch_size=PER_DEVICE_BATCH_SIZE,
        accumulation=GRADIENT_ACCUMULATION_STEPS,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
        run_common_evaluation=True,
    )
    print("Final registered run:", final_manifest["run_id"])
    print("Output directory:", final_manifest["run_dir"])
    print("Evaluation:", final_manifest.get("final_evaluation_metrics"))
    print("\nFINAL TRAINING COMPLETE")
    print("\nModel:", MODEL_ID)
    print("Run ID:", final_manifest["run_id"])
    print("Best checkpoint:", final_manifest["checkpoint_best_map"])
    print("Last checkpoint:", final_manifest["checkpoint_last"])
    print("Best mAP50-95:", final_manifest.get("best_validation_map"))
    print("Best APtiny:", final_manifest.get("best_validation_aptiny"))
    print("Training time:", final_manifest.get("total_training_seconds"), "seconds")
    print(
        "Evaluation command:",
        f'python scripts/evaluate.py --drive-root "{DRIVE_ROOT}" '
        f'--dataset-track 2class --split val --run-id "{final_manifest["run_id"]}" '
        "--resolutions 640",
    )
    print("Next notebook:", REPO_DIR / "notebooks" / "07_evaluate_all_models.ipynb")
else:
    print("Expensive stage is OFF; no model weights were changed.")
